# 03. 급등 가능 구간과 ENTER_NOW target 분리

기존 multi-horizon positive는 하나도 제거하지 않고 `ZONE` target으로 유지한다. 동일
positive episode 안에서는 미래 TP 전 downside와 대기 시간을 함께 고려해 한 행만
`ENTER_NOW` anchor로 지정한다.

- `target_zone_5m`: 기존 net +3% within 5m, 모든 인접 positive 유지
- `target_enter_first`: episode에서 가장 이른 유효 진입
- `target_enter_now`: `TP 전 downside + 시간비용`이 가장 낮은 위험조정 진입
- 미래 정보는 target 생성에만 쓰고 모델 feature에는 넣지 않는다.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd


def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'AGENT.md').is_file() and (candidate / 'README.md').is_file():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾지 못했습니다.')


PROJECT_ROOT = find_project_root()
PREPROCESS_ROOT = (PROJECT_ROOT / '../../results/preprocessing').resolve()
BASE_VERSION = 'scalp_30m_ohlcv_net3_multihorizon_5m_v2'
VERSION = 'scalp_30m_ohlcv_zone_entry_5m_v3'
WAIT_COST_PER_MINUTE = 0.002
TARGET_RETURN = 0.03

base_schema = json.loads((PREPROCESS_ROOT / f'{BASE_VERSION}_schema.json').read_text())
metadata = pd.read_parquet(PREPROCESS_ROOT / f'{BASE_VERSION}_metadata.parquet')
with np.load(PREPROCESS_ROOT / f'{BASE_VERSION}_features.npz') as loaded:
    horizon_targets = loaded['horizon_targets'].astype(np.int8)
    at_horizon_targets = loaded['at_horizon_targets'].astype(np.int8)
    hazard_events = loaded['hazard_events'].astype(np.int8)
    hazard_at_risk = loaded['hazard_at_risk'].astype(np.int8)

assert np.array_equal(metadata['feature_index'].to_numpy(), np.arange(len(metadata)))
assert np.array_equal(horizon_targets[:, -1], metadata['target_net3_within_5m'].to_numpy())
print('rows:', len(metadata), '| zone positives:', int(metadata['target_net3_within_5m'].sum()))

rows: 72181 | zone positives: 2847


## Risk-time entry quality

모든 ZONE positive는 순수익 +3%를 달성하므로, ENTER_NOW 안에서는 TP까지 기다리는 시간과
TP 전에 겪는 최대 하락만 비교한다.

\[
quality = 3\% - |pre\ hit\ downside| - 0.2\%	imes(first\ hit\ minute-2)
\]

episode별 quality 최대 행을 ENTER_NOW로 선택하고 동률이면 더 이른 판단 시점을 사용한다.

In [2]:
metadata = metadata.copy()
metadata['target_zone_5m'] = metadata['target_net3_within_5m'].astype(np.int8)
metadata['target_enter_first'] = np.zeros(len(metadata), dtype=np.int8)
metadata['target_enter_now'] = np.zeros(len(metadata), dtype=np.int8)
metadata['pre_hit_worst_net_return'] = np.nan
metadata['entry_wait_cost'] = np.nan
metadata['entry_quality'] = np.nan
metadata['entry_anchor_timestamp'] = pd.NaT
metadata['minutes_from_entry_anchor'] = np.nan

positive_indices = np.flatnonzero(metadata['target_zone_5m'].to_numpy() == 1)
for index in positive_indices:
    first_hit = int(metadata.at[index, 'first_hit_minute'])
    pre_hit_returns = [
        float(metadata.at[index, f'net_return_min_bid_{horizon}'])
        for horizon in range(1, first_hit)
    ]
    pre_hit_worst = min(pre_hit_returns) if pre_hit_returns else 0.0
    wait_cost = WAIT_COST_PER_MINUTE * max(first_hit - 2, 0)
    quality = TARGET_RETURN - max(-pre_hit_worst, 0.0) - wait_cost
    metadata.at[index, 'pre_hit_worst_net_return'] = pre_hit_worst
    metadata.at[index, 'entry_wait_cost'] = wait_cost
    metadata.at[index, 'entry_quality'] = quality

anchor_records = []
positive = metadata[metadata['target_zone_5m'].eq(1)]
for episode_id, episode in positive.groupby('positive_episode_id', sort=False):
    episode = episode.sort_values('decision_bar_timestamp')
    first_index = episode.index[0]
    quality_index = episode.sort_values(
        ['entry_quality', 'decision_bar_timestamp'], ascending=[False, True]
    ).index[0]
    anchor_timestamp = metadata.at[quality_index, 'decision_bar_timestamp']
    metadata.at[first_index, 'target_enter_first'] = 1
    metadata.at[quality_index, 'target_enter_now'] = 1
    metadata.loc[episode.index, 'entry_anchor_timestamp'] = anchor_timestamp
    metadata.loc[episode.index, 'minutes_from_entry_anchor'] = (
        episode['decision_bar_timestamp'] - anchor_timestamp
    ).dt.total_seconds().div(60).to_numpy()
    anchor_records.append({
        'positive_episode_id': episode_id,
        'session': metadata.at[quality_index, 'session'],
        'symbol': metadata.at[quality_index, 'symbol'],
        'episode_positive_samples': len(episode),
        'first_feature_index': int(metadata.at[first_index, 'feature_index']),
        'entry_feature_index': int(metadata.at[quality_index, 'feature_index']),
        'same_as_first': bool(first_index == quality_index),
        'first_timestamp': metadata.at[first_index, 'decision_bar_timestamp'],
        'entry_timestamp': anchor_timestamp,
        'first_hit_minute': int(metadata.at[first_index, 'first_hit_minute']),
        'entry_hit_minute': int(metadata.at[quality_index, 'first_hit_minute']),
        'first_pre_hit_worst': float(metadata.at[first_index, 'pre_hit_worst_net_return']),
        'entry_pre_hit_worst': float(metadata.at[quality_index, 'pre_hit_worst_net_return']),
        'entry_quality': float(metadata.at[quality_index, 'entry_quality']),
    })

anchors = pd.DataFrame(anchor_records)
print('zone positives:', int(metadata['target_zone_5m'].sum()))
print('episodes / ENTER_NOW:', len(anchors), int(metadata['target_enter_now'].sum()))
print('same as first:', f"{anchors['same_as_first'].mean():.2%}")

/tmp/ipykernel_2087994/2858258649.py:36: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2026-07-07 13:18:00+00:00' has dtype incompatible with datetime64[ns], please explicitly cast to a compatible dtype first.
  metadata.loc[episode.index, 'entry_anchor_timestamp'] = anchor_timestamp


zone positives: 2847
episodes / ENTER_NOW: 1128 1128
same as first: 44.86%


## Artifact 저장과 불변조건 감사

In [3]:
target_path = PREPROCESS_ROOT / f'{VERSION}_targets.npz'
metadata_path = PREPROCESS_ROOT / f'{VERSION}_metadata.parquet'
anchor_path = PREPROCESS_ROOT / f'{VERSION}_entry_anchors.parquet'
distribution_path = PREPROCESS_ROOT / f'{VERSION}_distribution.parquet'
schema_path = PREPROCESS_ROOT / f'{VERSION}_schema.json'

enter_now_target = metadata['target_enter_now'].to_numpy(np.int8)
enter_first_target = metadata['target_enter_first'].to_numpy(np.int8)
entry_quality = metadata['entry_quality'].to_numpy(np.float32)

assert len(metadata) == len(horizon_targets)
assert int(metadata['target_zone_5m'].sum()) == 2847
assert int(metadata['target_enter_now'].sum()) == len(anchors) == 1128
assert int(metadata['target_enter_first'].sum()) == len(anchors)
assert metadata.loc[metadata['target_enter_now'].eq(1), 'target_zone_5m'].eq(1).all()
assert metadata.loc[metadata['target_enter_first'].eq(1), 'target_zone_5m'].eq(1).all()
assert metadata.loc[metadata['target_zone_5m'].eq(1), 'positive_episode_id'].notna().all()
assert metadata.loc[metadata['target_zone_5m'].eq(1)].groupby('positive_episode_id')['target_enter_now'].sum().eq(1).all()
assert metadata.loc[metadata['target_zone_5m'].eq(1)].groupby('positive_episode_id')['target_enter_first'].sum().eq(1).all()

np.savez_compressed(
    target_path, horizon_targets=horizon_targets, at_horizon_targets=at_horizon_targets,
    hazard_events=hazard_events, hazard_at_risk=hazard_at_risk,
    enter_now_target=enter_now_target, enter_first_target=enter_first_target,
    entry_quality=entry_quality,
)
metadata.to_parquet(metadata_path, index=False, compression='zstd')
anchors.to_parquet(anchor_path, index=False, compression='zstd')
distribution = metadata.groupby('session').agg(
    samples=('sample_id', 'size'), symbols=('symbol', 'nunique'),
    zone_positives=('target_zone_5m', 'sum'), zone_rate=('target_zone_5m', 'mean'),
    entry_positives=('target_enter_now', 'sum'), entry_rate=('target_enter_now', 'mean'),
    first_entry_positives=('target_enter_first', 'sum'),
).reset_index()
distribution.to_parquet(distribution_path, index=False, compression='zstd')

schema = {
    'version': VERSION, 'base_preprocessing_version': BASE_VERSION,
    'sequence_artifact': base_schema['artifacts']['features'],
    'feature_names': base_schema['feature_names'],
    'default_feature_names': base_schema['default_feature_names'],
    'sequence_length': base_schema['sequence_length'],
    'zone_target': {
        'name': 'target_zone_5m', 'rule': base_schema['label']['positive_rule'],
        'adjacent_positive_policy': 'retain all 2,847 independently valid positive rows',
    },
    'entry_targets': {
        'primary': 'target_enter_now', 'comparison': 'target_enter_first',
        'one_anchor_per_positive_episode': True,
        'quality_formula': '0.03 - abs(min pre-hit min_bid return below zero) - 0.002 * max(first_hit_minute - 2, 0)',
        'future_data_usage': 'target construction only; never a model feature',
    },
    'artifacts': {
        'targets': str(target_path), 'metadata': str(metadata_path),
        'entry_anchors': str(anchor_path), 'distribution': str(distribution_path),
    },
}
schema_path.write_text(json.dumps(schema, ensure_ascii=False, indent=2), encoding='utf-8')

print('metadata:', metadata.shape)
display(distribution)
display(anchors[[
    'same_as_first', 'first_hit_minute', 'entry_hit_minute',
    'first_pre_hit_worst', 'entry_pre_hit_worst', 'entry_quality',
]].describe().T)

metadata: (72181, 84)


,session,samples,symbols,zone_positives,zone_rate,entry_positives,entry_rate,first_entry_positives
0,session_2026-07-07,4321,13,115,0.026614,47,0.010877,47
1,session_2026-07-08,10343,25,362,0.035000,152,0.014696,152
2,session_2026-07-09,10208,27,388,0.038009,165,0.016164,165
3,session_2026-07-10,10230,29,534,0.052199,219,0.021408,219
4,session_2026-07-13,7953,18,405,0.050924,147,0.018484,147
5,session_2026-07-14,7043,20,266,0.037768,99,0.014057,99
6,session_2026-07-15,9556,21,409,0.042800,162,0.016953,162
7,session_2026-07-16,5918,16,237,0.040047,81,0.013687,81
8,session_2026-07-17,6609,20,131,0.019821,56,0.008473,56


,count,mean,std,min,25%,50%,75%,max
first_hit_minute,1128.0,4.323582,0.939564,2.000000,4.000000,5.000000,5.000000,5.000000
entry_hit_minute,1128.0,3.375887,1.065147,2.000000,2.000000,3.000000,4.000000,5.000000
first_pre_hit_worst,1128.0,-0.014366,0.013864,-0.147500,-0.018224,-0.009607,-0.006079,-0.000084
entry_pre_hit_worst,1128.0,-0.007900,0.007287,-0.080922,-0.009291,-0.006374,-0.003526,-0.000056
entry_quality,1128.0,0.019348,0.007897,-0.054922,0.017127,0.021169,0.023802,0.029923
